# To TG


# INIT


In [ ]:
import numpy as np
from pprint import pprint
from copy import deepcopy
import os
import sys
from pathlib import Path

from laboneq.contrib.example_helpers.generate_descriptor import generate_descriptor
from laboneq.dsl.calibration import Oscillator, SignalCalibration
from laboneq.dsl.device import DeviceSetup
from laboneq.dsl.enums import ModulationType
from laboneq.simple import *

repo_root = next(
    (
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "configs").exists() and (p / "qpu_parameters").exists() and (p / "notebooks").exists()
    ),
    Path.cwd(),
)
for path in [repo_root, repo_root / "qubit-experiment", repo_root.parent / "qubit-experiment"]:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

# descriptor = generate_descriptor(...)
# setup = DeviceSetup.from_descriptor(yaml_text=descriptor, server_host="192.168.0.83")
descriptor_candidates = [
    repo_root / "configs/descriptors/1port.yaml",
    Path("configs/descriptors/1port.yaml"),
    Path("../configs/descriptors/1port.yaml"),
]
descriptor_path = next((p for p in descriptor_candidates if p.exists()), descriptor_candidates[0])
setup = DeviceSetup.from_yaml(filepath=str(descriptor_path.resolve()), server_host="192.168.0.83")
setup.instruments[0].device_options = "SHFQC/PLUS/QC6CH"

bus_ids = [f"b{i}" for i in range(3)]
bus_ports = [3, 4, 5]  # used 1,2,3 for qubit drive
for port, bus in zip(bus_ports, bus_ids):
    setup.add_connections(
        setup.instruments[0].uid,
        create_connection(
            to_signal=f"{bus}/drive",
            ports=f"SGCHANNELS/{port}/OUTPUT",
        ),
    )

# Calibrate qubit drive/measure lines for oscillator phase increments.
qubit_ids = [uid for uid in setup.logical_signal_groups if uid.startswith("q")]
for qubit in qubit_ids:
    for line, frequency, mod_type in [
        ("drive", 5e9, ModulationType.HARDWARE),
        ("drive_ef", 6e9, ModulationType.HARDWARE),
        ("measure", 4e9, ModulationType.SOFTWARE),
    ]:
        logical_signal = setup.logical_signal_by_uid(f"{qubit}/{line}")
        oscillator = Oscillator(modulation_type=mod_type)
        logical_signal.calibration = SignalCalibration(
            local_oscillator=Oscillator(frequency=frequency),
            oscillator=oscillator,
        )
        if line == "measure":
            acquire_signal = setup.logical_signal_by_uid(f"{qubit}/acquire")
            acquire_signal.calibration = SignalCalibration(
                local_oscillator=Oscillator(frequency=frequency),
                oscillator=oscillator,
            )

from qubit_experiment.qpu_types.fixed_transmon import FixedTransmonQubit
from qubit_experiment.qpu_types.bus_cavity import BusCavity
from qubit_experiment.qpu_types.fixed_transmon.operations import FixedTransmonOperations
from qubit_experiment.qpu_types.bus_cavity.operations import BusCavityOperations
from laboneq.dsl.quantum.qpu import QPU

qubit_uids = [uid for uid in setup.logical_signal_groups if uid.startswith("q")]
bus_uids = [uid for uid in setup.logical_signal_groups if uid.startswith("b")]

qubits = FixedTransmonQubit.from_device_setup(setup, qubit_uids=qubit_uids)
buses = BusCavity.from_device_setup(setup, qubit_uids=bus_uids)
qpu = QPU(
    quantum_elements={"qubits": qubits, "bus": buses},
    quantum_operations=[FixedTransmonOperations, BusCavityOperations],
)

from datetime import datetime

def find_latest_json(folder_path):
    files = [f for f in os.listdir(folder_path)]
    timestamps = []
    for file in files:
        timestamp_str = file.split("_", 1)[0]
        for fmt in ("%Y%m%d-%H%M%S", "%Y%m%d-%H%M"):
            try:
                timestamp = datetime.strptime(timestamp_str, fmt)
                timestamps.append((timestamp, file))
                break
            except ValueError:
                continue

    if timestamps:
        latest_file = max(timestamps, key=lambda x: x[0])[1]
        return os.path.join(folder_path, latest_file)
    return None

qb_pars_file = find_latest_json(str(repo_root / "qpu_parameters"))
print(f"LOADED: {qb_pars_file}")

import __main__
import importlib
import laboneq.dsl.quantum.qpu as qpu_mod

class CombinedOperations(FixedTransmonOperations, BusCavityOperations):
    pass

qpu_mod.CombinedOperations = CombinedOperations
__main__.CombinedOperations = CombinedOperations

for legacy_name, canonical_name in {
    "qpu_types": "qubit_experiment.qpu_types",
    "qpu_types.fixed_transmon": "qubit_experiment.qpu_types.fixed_transmon",
    "qpu_types.fixed_transmon.qubit_types": "qubit_experiment.qpu_types.fixed_transmon.qubit_types",
    "qpu_types.fixed_transmon.operations": "qubit_experiment.qpu_types.fixed_transmon.operations",
    "qpu_types.bus_cavity": "qubit_experiment.qpu_types.bus_cavity",
    "qpu_types.bus_cavity.bus_types": "qubit_experiment.qpu_types.bus_cavity.bus_types",
    "qpu_types.bus_cavity.operations": "qubit_experiment.qpu_types.bus_cavity.operations",
}.items():
    sys.modules.setdefault(legacy_name, importlib.import_module(canonical_name))

qpu = load(qb_pars_file)

buses = qpu.groups.bus
qubits = qpu.groups.qubits

from laboneq.simple import workflow
folder_store = workflow.logbook.FolderStore(str(repo_root / "experiment_store"))
folder_store.activate()


In [ ]:
from laboneq.simple import Session
session = Session(setup)
session.connect(ignore_version_mismatch=False, do_emulation=False)
#session.disconnect()


## parameter

In [ ]:

qubits[0].parameters.ge_drive_amplitude_pi = 0.6460
qubits[0].parameters.ge_drive_amplitude_pi2 = 0.6450

qubits[1].parameters.ge_drive_amplitude_pi =  0.4580
qubits[1].parameters.ge_drive_amplitude_pi2 = 0.4580

qubits[2].parameters.ge_drive_amplitude_pi = 0.8070
qubits[2].parameters.ge_drive_amplitude_pi2 = 0.8070

## RAMSEY

In [ ]:
from qubit_experiment.experiments import ramsey


q = qubits[2]
temporary_parameters = {}
temp_pars =deepcopy(q.parameters)
# temp_pars = qubits[0].parameters
# temp_pars.resonance_frequency_ge = 4.9528e9
# temp_pars.readout_length = 1.2e-6
# temp_pars.readout_amplitude=0.65
temp_pars.readout_length = 1.2e-6
temp_pars.readout_integration_length = 1.2e-6
temporary_parameters[q.uid] = temp_pars


#######################################################################
delays = np.linspace(0,15e-6,151)
detunings = 0.3e6,
pprint(q.readout_parameters())
#######################################################################
options = ramsey.experiment_workflow.options()
options.update(False)
options.count(1024)
options.use_cal_traces(True)
# Build and run Ramsey workflow (no explicit command table options required)
ramsey_wf = ramsey.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=q,
    delays=delays,
    detunings=detunings,
    options=options,
    temporary_parameters=temporary_parameters
)
ramsey_result = ramsey_wf.run()
#qubit_spec_compiled = session.compile(amplitude_rabi.create_experiment(qpu=qpu,qubit=q,amplitudes=amplitudes, options=options))
print(ramsey_result.tasks['analysis_workflow'].output)

## X90

In [ ]:
from qubit_experiment.experiments import amplitude_fine
q = qubits[2]
temporary_parameters = {}
temp_pars =deepcopy(q.parameters)
# temp_pars = qubits[2].parameters

# temp_pars.readout_length = 1.4e-6
temp_pars.readout_integration_length = 1800e-9

temporary_parameters[q.uid] = temp_pars
#######################################################################
repetitions =np.arange(1,10)
#######################################################################
options = amplitude_fine.experiment_workflow_x90.options()
options.update(False) 
options.use_cal_traces(True)
options.do_pca(False)
options.count(1024)

#print(workflow.show_fields(options))

###################################################################
error_amp_half = amplitude_fine.experiment_workflow_x90(
    session=session,
    qpu=qpu,
    qubits=q,
    repetitions=repetitions,
    temporary_parameters=temporary_parameters,
    options=options
)

error_amp_half_result = error_amp_half.run()
#qubit_spec_compiled = session.compile(amplitude_rabi.create_experiment(qpu=qpu,qubit=q,amplitudes=amplitudes, options=options))
print(error_amp_half_result.tasks['analysis_workflow'].output)

## X180

In [ ]:
from qubit_experiment.experiments import amplitude_fine
q = qubits[2]
temporary_parameters = {}
temp_pars =deepcopy(q.parameters)
# temp_pars = qubits[2].parameters
# temp_pars.readout_amplitude=0.5
# temp_pars.readout_length = 1.4e-6
temp_pars.readout_integration_length = 1.8e-6

temporary_parameters[q.uid] = temp_pars

#######################################################################
repetitions =np.arange(1,14) # Due to short T1, 20   
#######################################################################
options = amplitude_fine.experiment_workflow_x180.options()
options.update(False)
options.use_cal_traces(True)
#options.do_pca(False)

#print(workflow.show_fields(options))

###################################################################
error_amp = amplitude_fine.experiment_workflow_x180(
    session=session,
    qpu=qpu,
    qubits=q,
    repetitions=repetitions,
    temporary_parameters=temporary_parameters,
    options=options
)

error_amp_result = error_amp.run()
#qubit_spec_compiled = session.compile(amplitude_rabi.create_experiment(qpu=qpu,qubit=q,amplitudes=amplitudes, options=options))
print(error_amp_result.tasks['analysis_workflow'].output)

# Multiplexed IQ Clouds


In [ ]:
from qubit_experiment.experiments import iq_cloud_common, iq_cloud
q = qubits[0]
qq = qubits[2]

temporary_parameters = {}
q_temp_pars = deepcopy(q.parameters)
qq_temp_pars = deepcopy(qq.parameters)

# q_temp_pars.readout_integration_delay = 180e-9
# qq_temp_pars.readout_integration_delay= 180e-9
# q_temp_pars.readout_length = 1.6e-6
# qq_temp_pars.readout_length = 1.6e-6
temporary_parameters[q.uid] = q_temp_pars
temporary_parameters[qq.uid] = qq_temp_pars
#######################################################################

options = iq_cloud.experiment_workflow.options()
options.do_analysis(True)
options.update(False)
#options.count(1024)

iq_result = iq_cloud.experiment_workflow(
    session=session,
    qpu=qpu,
    qubits=[q,qq],
    temporary_parameters=temporary_parameters,
    options=options,
).run()


# Bus setting


# ToTG: 2Q/3Q Quantum State Tomography + Custom Bus Drive 실행 예제

이 노트북은 최신 canonical QST 모듈을 기준으로 아래를 한 번에 보여줍니다.

1. 2Q tomography 실행 (`qubit_experiment.experiments.twoq_qst`)
2. custom bus drive / spectator 실험 실행 (`custom_qubit_experiment.custom_experiments.two_qubit_state_tomography_3`)
3. 3Q tomography 실행 (`qubit_experiment.experiments.threeq_qst`)
4. 결과에서 density matrix(`rho_hat`) 및 요약 payload 확인

대상 독자: 이미 `session`, `qpu`, `qubits`, `buses`를 준비해 둔 실험 사용자


## 최신 실험 모듈 사용법 (핵심)

### 1) 2Q canonical entry point
- `twoq_qst.run_bundle(...)`
- `twoq_qst.experiment_workflow(...)`

필수 인자:
- `session`: LabOne Q 세션
- `qpu`: `QPU` 객체
- `qubits`: tomography 대상 2개 qubit 리스트
- `bus`: bus element

주요 선택 인자:
- `readout_calibration_result`: readout calibration 결과
- `target_state`: fidelity 계산 target (`"++"`, `"00"`, `"+-"` 등)
- `temporary_parameters`: 임시 파라미터 오버라이드
- `options`: workflow 옵션 객체
- `analysis_options`: analysis workflow 옵션 객체 (`run_bundle` 사용 시)

### 2) 옵션 객체
- `opts = twoq_qst.experiment_workflow.options()`
- `analysis_opts = twoq_qst_analysis.analysis_workflow.options()`

### 3) 3Q canonical entry point
- `threeq_qst.run_bundle(...)`
- `threeq_qst.experiment_workflow(...)`


In [ ]:
from qubit_experiment.experiments import two_qubit_readout_calibration
from qubit_experiment.experiments import twoq_qst

print("repo_root:", repo_root)


## 실행 전 환경 객체 확인

이 셀은 아래 전역 변수가 이미 준비되어 있다는 가정으로 동작합니다.
- `session`
- `qpu`
- `qubits`
- `buses`

보통 `2Q_QST_TEST_NEW.ipynb`의 INIT/장비연결 셀을 먼저 실행한 뒤 이 노트북을 사용합니다.


In [ ]:
required = ["session", "qpu", "qubits", "buses"]
missing = [name for name in required if name not in globals()]
if missing:
    raise RuntimeError(
        "Missing required globals: "
        + ", ".join(missing)
        + ". Run your setup/connection notebook first."
    )

# Example target qubits/buses (필요시 인덱스 변경)
ctrl = qubits[0]
targ = qubits[2]
custom_buses = [buses[0], buses[1], buses[2]]

if "temporary_parameters" not in globals():
    temporary_parameters = {}

print("ctrl:", ctrl.uid)
print("targ:", targ.uid)
print("bus list:", [b.uid for b in custom_buses])


## custom bus drive 파라미터 설정

아래 예제는 각 bus에 RIP 관련 파라미터를 직접 주입합니다.

필드 의미:
- `resonance_frequency_bus`: bus 공진 주파수
- `rip_detuning`: RIP detuning
- `rip_amplitude`: RIP pulse amplitude
- `rip_length`: RIP pulse length
- `rip_phase`: RIP phase

실험 모듈은 이 값들을 읽어 bus drive를 구성합니다.


In [ ]:
buses[0].parameters.drive_lo_frequency = 5.6e9 #b1
buses[1].parameters.drive_lo_frequency = 6.2e9 #b3
buses[2].parameters.drive_lo_frequency = 6.2e9 #b2

buses[0].parameters.resonance_frequency_bus = 5.5055e9
buses[1].parameters.resonance_frequency_bus = 6.4214e9
buses[2].parameters.resonance_frequency_bus = 5.9807e9

buses[0].parameters.rip_detuning = 12e6
buses[1].parameters.rip_detuning = -60e6
buses[2].parameters.rip_detuning = -60e6

buses[0].parameters.rip_amplitude = 0.42
buses[1].parameters.rip_amplitude = 0.0
buses[2].parameters.rip_amplitude = 0.0

t=432e-9
buses[0].parameters.rip_length = t
buses[1].parameters.rip_length = t
buses[2].parameters.rip_length = t


#drive_frequency_bus = self.resonance_frequency_bus + self.rip_detuning - self.drive_lo_frequency
print(buses[0].parameters.drive_frequency_bus + buses[0].parameters.drive_lo_frequency) #
print(buses[1].parameters.drive_frequency_bus + buses[1].parameters.drive_lo_frequency)
print(buses[2].parameters.drive_frequency_bus + buses[2].parameters.drive_lo_frequency)


print(buses[0].parameters) #mode 1
print(buses[1].parameters) #mode 3
print(buses[2].parameters) #mode 2

## QST 실행 (RIP on + 외부 readout calibration 사용)

권장 패턴:
1. readout calibration을 먼저 1회 수행
2. 결과를 `readout_calibration_result`로 tomography workflow에 전달
3. tomography 옵션에서 `do_readout_calibration(False)`로 고정

이 패턴을 쓰면 shot sweep/반복 실험에서 calibration 누락 문제를 줄일 수 있습니다.


In [ ]:
from qubit_experiment.experiments import two_qubit_readout_calibration
from custom_qubit_experiment.custom_experiments import two_qubit_state_tomography_3

qubits[0].parameters.readout_integration_delay = 180e-9
qubits[1].parameters.readout_integration_delay = 180e-9
qubits[2].parameters.readout_integration_delay = 180e-9

ctrl = qubits[1]
targ = qubits[2]
spec = qubits[0]
bus = [buses[0],buses[1],buses[2]]

# ✅ 컴파일러 충돌 방지: temporary_parameters 내부의 delay 값을 강제로 완벽히 동기화
for q in [ctrl, targ, spec]:
    if q.uid in temporary_parameters:
        temporary_parameters[q.uid].readout_integration_delay = 180e-9
# 강제로 재실행
readout_cal_result_new = None
# 1) Readout calibration (없으면 1회 생성)
if "readout_cal_result_new" not in globals() or readout_cal_result_new is None:
    readout_cal_result_new = two_qubit_readout_calibration.experiment_workflow(
        session=session,
        qpu=qpu,
        # ctrl=qubits[2],
        # targ=qubits[0],
        qubits=[ctrl, targ],
        temporary_parameters=temporary_parameters,
    ).run()

# 2) Tomography options
opts = two_qubit_state_tomography_3.experiment_workflow.options()
opts.do_analysis(True)
opts.do_readout_calibration(True)
opts.validation_mode(False)
opts.use_rip(True)
opts.initial_state("++")
opts.enforce_target_match(True)
opts.count(2048)

# 3) Tomography run (custom bus list 전달)
twoq_qst_result = two_qubit_state_tomography_3.experiment_workflow(
    session=session,
    qpu=qpu,
    ctrl=ctrl,
    targ=targ,
    spec=spec,
    # bus=bus,
    bus = bus,
    readout_calibration_result=readout_cal_result_new,
    target_state="++",
    targ_detuning=-2.14e6,  # ← Ramsey detuning
    spec_prep="nop",   
    options=opts,
    temporary_parameters=temporary_parameters,
).run()

# twoq_qst_result = two_qubit_state_tomography_2.experiment_workflow(
#     session=session,
#     qpu=qpu,
#     ctrl=qubits[0],
#     targ=qubits[2],
#     bus=[buses[0], buses[1], buses[2]],
#     spec=qubits[1],              # ✅ spectator qubit
#     spec_prep="pi",              # ✅ "nop" or "pi"
#     targ_detuning=-3.255e6,
#     readout_calibration_result=readout_cal_result_new,
#     target_state="++",
#     options=opts,
#     temporary_parameters=temporary_parameters,
# ).run()

print("QST workflow finished.")


In [ ]:
from laboneq.simple import show_pulse_sheet
from laboneq.contrib.example_helpers.plotting.plot_helpers import plot_simulation

show_pulse_sheet(compiled_experiment=twoq_qst_result.tasks["compile_experiment"].output,name='test', interactive=True)
# plot_simulation(compiled_experiment=twoq_qst_result.tasks["compile_experiment"].output)
plot_simulation(
    compiled_experiment=twoq_qst_result.tasks["compile_experiment"].output,
    start_time=0.0e-6,  # 시작 시간
    length=3e-6       # 그 시점부터 볼 길이
)

In [ ]:
"""Plot density matrices from 3-bus RIP tomography results.

Usage:
    After running rip_3bus_tomography, pass the results to these functions.
    Supports side-by-side comparison of spec=nop vs spec=pi.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import Normalize
from qubit_experiment.experiments import twoq_qst

# ── Circuit QED visualization settings ──────────────────────────────────
rcParams['figure.figsize'] = (10, 7)
rcParams['figure.dpi'] = 150
rcParams['font.size'] = 18
rcParams['axes.labelsize'] = 20
rcParams['axes.titlesize'] = 22
rcParams['xtick.labelsize'] = 16
rcParams['ytick.labelsize'] = 16
rcParams['legend.fontsize'] = 16
rcParams['lines.linewidth'] = 3.0
rcParams['lines.markersize'] = 10
rcParams['axes.linewidth'] = 2.0
rcParams['xtick.major.width'] = 2.0
rcParams['ytick.major.width'] = 2.0
rcParams['xtick.major.size'] = 8
rcParams['ytick.major.size'] = 8
rcParams['grid.linewidth'] = 1.5
rcParams['grid.alpha'] = 0.3
rcParams['image.cmap'] = 'viridis'


# ── 2-qubit basis labels ───────────────────────────────────────────────
BASIS_LABELS_2Q = [r'$|00\rangle$', r'$|01\rangle$', r'$|10\rangle$', r'$|11\rangle$']

def unwrap_output(obj):
    cur = obj
    for _ in range(24):
        if hasattr(cur, "output"):
            cur = cur.output
            continue
        return cur
    return cur


def _is_reference_like(obj):
    return obj is not None and obj.__class__.__name__ == "Reference"


def _contains_reference(obj, depth=0, max_depth=12):
    if depth > max_depth:
        return False
    cur = unwrap_output(obj)
    if _is_reference_like(cur):
        return True
    if isinstance(cur, dict):
        return any(_contains_reference(v, depth + 1, max_depth) for v in cur.values())
    if isinstance(cur, (list, tuple)):
        return any(_contains_reference(v, depth + 1, max_depth) for v in cur)
    return False


def _iter_tasks(node):
    tasks = getattr(node, "tasks", None)
    if tasks is None:
        return []
    try:
        return list(tasks)
    except Exception:
        return []


def _task_output(tasks, key):
    try:
        out = unwrap_output(tasks[key].output)
    except Exception:
        return None
    if _is_reference_like(out):
        return None
    return out


def _assemble_from_analysis_tasks(analysis_node):
    tasks = getattr(analysis_node, "tasks", None)
    if tasks is None:
        return None

    assignment = _task_output(tasks, "extract_assignment_matrix")
    tomography_counts = _task_output(tasks, "collect_tomography_counts")
    mle = _task_output(tasks, "maximum_likelihood_reconstruct")
    state_metrics = _task_output(tasks, "calculate_state_metrics")
    discriminator = _task_output(tasks, "fit_discriminator_from_readout_calibration")

    if not all(isinstance(x, dict) for x in [assignment, tomography_counts, mle, state_metrics, discriminator]):
        return None

    assembled = {
        "assignment_matrix": assignment.get("assignment_matrix"),
        "assignment_counts": assignment.get("counts_matrix_soft"),
        "assignment_counts_soft": assignment.get("counts_matrix_soft"),
        "assignment_counts_hard": assignment.get("counts_matrix_hard"),
        "tomography_counts": tomography_counts.get("counts"),
        "tomography_counts_hard": tomography_counts.get("counts_hard"),
        "setting_labels": tomography_counts.get("setting_labels"),
        "shots_per_setting": tomography_counts.get("shots_per_setting"),
        "rho_hat_real": mle.get("rho_hat_real"),
        "rho_hat_imag": mle.get("rho_hat_imag"),
        "predicted_probabilities": mle.get("predicted_probabilities"),
        "predicted_counts": mle.get("predicted_counts"),
        "optimizer_success": mle.get("optimizer_success"),
        "optimizer_message": mle.get("optimizer_message"),
        "negative_log_likelihood": mle.get("negative_log_likelihood"),
        "metrics": state_metrics,
        "discriminator_model": discriminator.get("model"),
        "classification_diagnostics": discriminator.get("diagnostics"),
        "bitflip_ctrl": False,
        "bitflip_targ": False,
    }
    return assembled


def _find_analysis_node(root, depth=0, max_depth=12):
    if root is None or depth > max_depth:
        return None
    tasks = _iter_tasks(root)
    if tasks:
        names = {getattr(t, "name", "") for t in tasks}
        if "maximum_likelihood_reconstruct" in names and "extract_assignment_matrix" in names:
            return root
        for t in tasks:
            found = _find_analysis_node(t, depth + 1, max_depth)
            if found is not None:
                return found
            found = _find_analysis_node(getattr(t, "output", None), depth + 1, max_depth)
            if found is not None:
                return found
    out = getattr(root, "output", None)
    if out is not None and out is not root:
        return _find_analysis_node(out, depth + 1, max_depth)
    return None

# ── Helper: extract rho from workflow result ────────────────────────────
def extract_analysis_output(workflow_result):
    analysis_node = _find_analysis_node(workflow_result)
    if analysis_node is None:
        raise RuntimeError("Could not locate analysis workflow node.")

    out = unwrap_output(getattr(analysis_node, "output", None))
    if isinstance(out, dict) and not _contains_reference(out):
        return out

    assembled = _assemble_from_analysis_tasks(analysis_node)
    if isinstance(assembled, dict):
        return assembled

    raise RuntimeError("Could not materialize concrete analysis output from workflow tasks.")


def extract_rho(result):
    """Extract density matrix from tomography workflow result.

    Tries multiple paths depending on the analysis output structure.
    Returns a 4×4 complex numpy array.
    """
    out = result
    # Unwrap .output if present
    for _ in range(5):
        if hasattr(out, 'output'):
            out = out.output
        else:
            break

    if isinstance(out, dict):
        # Path 1: top-level analysis_result
        ar = out.get("analysis_result", out)
        if hasattr(ar, 'output'):
            ar = ar.output
        if isinstance(ar, dict):
            # Path 1a: metrics → rho
            metrics = ar.get("metrics", {})
            if isinstance(metrics, dict) and "rho" in metrics:
                return np.array(metrics["rho"])
            # Path 1b: direct rho key
            if "rho" in ar:
                return np.array(ar["rho"])
            # Path 1c: rho_mle
            if "rho_mle" in ar:
                return np.array(ar["rho_mle"])
            # Path 1d: reconstructed_state
            if "reconstructed_state" in ar:
                return np.array(ar["reconstructed_state"])

    raise ValueError(
        "Could not extract density matrix from result. "
        "Check the analysis output structure with: result.output"
    )


# ── 3D bar plot of density matrix ──────────────────────────────────────

def plot_density_matrix_3d(
    rho,
    title="Density Matrix",
    ax=None,
    figsize=(10, 8),
    bar_width=0.4,
    cmap="viridis",
):
    """3D bar plot of a density matrix (real + imaginary parts).

    Parameters
    ----------
    rho : np.ndarray
        4×4 complex density matrix.
    title : str
        Plot title.
    ax : tuple of (ax_real, ax_imag) or None
        If None, creates a new figure with 2 subplots.
    figsize : tuple
        Figure size (only used if ax is None).
    bar_width : float
        Width of each bar.
    cmap : str
        Colormap name.

    Returns
    -------
    fig, (ax_real, ax_imag)
    """
    rho = np.array(rho)
    dim = rho.shape[0]
    labels = BASIS_LABELS_2Q[:dim]

    if ax is None:
        fig = plt.figure(figsize=(figsize[0] * 2, figsize[1]))
        ax_real = fig.add_subplot(121, projection='3d')
        ax_imag = fig.add_subplot(122, projection='3d')
    else:
        ax_real, ax_imag = ax
        fig = ax_real.get_figure()

    for ax_part, data, part_label in [
        (ax_real, rho.real, r"Re($\rho$)"),
        (ax_imag, rho.imag, r"Im($\rho$)"),
    ]:
        _draw_3d_bars(ax_part, data, labels, bar_width, cmap, part_label)

    fig.suptitle(title, fontsize=24, fontweight='bold', y=1.02)
    fig.tight_layout()
    return fig, (ax_real, ax_imag)


def _draw_3d_bars(ax, data, labels, bar_width, cmap, zlabel):
    """Draw 3D bar chart on a given axes."""
    dim = data.shape[0]
    xpos, ypos = np.meshgrid(np.arange(dim), np.arange(dim), indexing='ij')
    xpos = xpos.flatten()
    ypos = ypos.flatten()
    zpos = np.zeros_like(xpos)
    dz = data.flatten()

    # Color by value
    vmax = max(abs(dz.max()), abs(dz.min()), 0.5)
    norm = Normalize(vmin=-vmax, vmax=vmax)
    colors = plt.cm.get_cmap(cmap)(norm(dz))

    ax.bar3d(
        xpos - bar_width / 2,
        ypos - bar_width / 2,
        zpos,
        bar_width,
        bar_width,
        dz,
        color=colors,
        edgecolor='black',
        linewidth=0.8,
        alpha=0.9,
    )

    ax.set_xticks(range(dim))
    ax.set_yticks(range(dim))
    ax.set_xticklabels(labels, fontsize=14)
    ax.set_yticklabels(labels, fontsize=14)
    ax.set_zlabel(zlabel, fontsize=18, labelpad=12)
    ax.set_zlim(-1.0, 1.0)
    ax.tick_params(axis='z', labelsize=14, pad=5)
    ax.view_init(elev=25, azim=-55)


# ── 2D heatmap (hinton-style) of density matrix ────────────────────────

def plot_density_matrix_2d(
    rho,
    title="Density Matrix",
    figsize=(16, 6),
):
    """2D heatmap of real and imaginary parts of the density matrix.

    Parameters
    ----------
    rho : np.ndarray
        4×4 complex density matrix.
    title : str
        Plot title.
    figsize : tuple
        Figure size.

    Returns
    -------
    fig, (ax_real, ax_imag)
    """
    rho = np.array(rho)
    dim = rho.shape[0]
    labels = BASIS_LABELS_2Q[:dim]

    fig, (ax_real, ax_imag) = plt.subplots(1, 2, figsize=figsize)

    vmax = max(np.abs(rho.real).max(), np.abs(rho.imag).max(), 0.5)

    for ax, data, part_label in [
        (ax_real, rho.real, r"Re($\rho$)"),
        (ax_imag, rho.imag, r"Im($\rho$)"),
    ]:
        im = ax.imshow(
            data, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
            interpolation='nearest', aspect='equal',
        )
        ax.set_xticks(range(dim))
        ax.set_yticks(range(dim))
        ax.set_xticklabels(labels, fontsize=16)
        ax.set_yticklabels(labels, fontsize=16)
        ax.set_title(part_label, fontsize=20)

        # Annotate values
        for i in range(dim):
            for j in range(dim):
                val = data[i, j]
                color = 'white' if abs(val) > vmax * 0.6 else 'black'
                ax.text(j, i, f'{val:.3f}', ha='center', va='center',
                        fontsize=14, fontweight='bold', color=color)

        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=14, width=2, length=6)

    fig.suptitle(title, fontsize=24, fontweight='bold')
    fig.tight_layout()
    return fig, (ax_real, ax_imag)


# ── Side-by-side comparison: spec=nop vs spec=pi ───────────────────────

def plot_comparison(
    rho_nop,
    rho_pi,
    rip_length=None,
    plot_type="2d",
):
    """Compare density matrices for spectator nop vs pi.

    Parameters
    ----------
    rho_nop : np.ndarray
        Density matrix with spectator in ground state.
    rho_pi : np.ndarray
        Density matrix with spectator in excited state.
    rip_length : float or None
        RIP duration in seconds (for title annotation).
    plot_type : str
        "2d" for heatmap, "3d" for bar plot.

    Returns
    -------
    dict with figures
    """
    rho_nop = np.array(rho_nop)
    rho_pi = np.array(rho_pi)
    rho_diff = rho_pi - rho_nop

    length_str = ""
    if rip_length is not None:
        length_str = f" (RIP = {rip_length * 1e9:.0f} ns)"

    plot_func = plot_density_matrix_2d if plot_type == "2d" else plot_density_matrix_3d

    fig_nop, _ = plot_func(rho_nop, title=f"Spec = ground{length_str}")
    fig_pi, _ = plot_func(rho_pi, title=f"Spec = excited{length_str}")
    fig_diff, _ = plot_func(rho_diff, title=f"Difference (excited − ground){length_str}")

    return {
        "fig_nop": fig_nop,
        "fig_pi": fig_pi,
        "fig_diff": fig_diff,
    }


# ── Metrics summary ────────────────────────────────────────────────────

def print_metrics(result, label=""):
    """Print key tomography metrics from a workflow result."""
    out = result
    for _ in range(5):
        if hasattr(out, 'output'):
            out = out.output
        else:
            break

    if not isinstance(out, dict):
        print(f"[{label}] Could not extract metrics.")
        return

    ar = out.get("analysis_result", out)
    if hasattr(ar, 'output'):
        ar = ar.output
    if not isinstance(ar, dict):
        print(f"[{label}] Could not extract analysis_result.")
        return

    metrics = ar.get("metrics", {})
    if not isinstance(metrics, dict):
        metrics = {}

    print(f"\n{'='*50}")
    print(f"  {label}")
    print(f"{'='*50}")
    fid = metrics.get("fidelity_to_target")
    if fid is not None:
        print(f"  Fidelity to target : {fid:.4f}")
    purity = metrics.get("purity")
    if purity is not None:
        print(f"  Purity             : {purity:.4f}")
    concurrence = metrics.get("concurrence")
    if concurrence is not None:
        print(f"  Concurrence        : {concurrence:.4f}")
    ent_form = metrics.get("entanglement_of_formation")
    if ent_form is not None:
        print(f"  Ent. of Formation  : {ent_form:.4f}")
    print(f"{'='*50}\n")


# ═══════════════════════════════════════════════════════════════════════
# Example usage (paste in your notebook after running tomography)
# ═══════════════════════════════════════════════════════════════════════
#

analysis_nop = extract_analysis_output(twoq_qst_result)
rho_nop = np.asarray(analysis_nop["rho_hat_real"], dtype=float) + \
          1j * np.asarray(analysis_nop["rho_hat_imag"], dtype=float)
rho_nop = (rho_nop + rho_nop.conj().T) / 2.0
rho_nop /= np.trace(rho_nop)

# analysis_pi = extract_analysis_output(result_spec_pi)
# rho_pi = np.asarray(analysis_pi["rho_hat_real"], dtype=float) + \
#          1j * np.asarray(analysis_pi["rho_hat_imag"], dtype=float)
# rho_pi = (rho_pi + rho_pi.conj().T) / 2.0
# rho_pi /= np.trace(rho_pi)



# from plot_rip_tomography import (
#     extract_rho, plot_density_matrix_2d, plot_density_matrix_3d,
#     plot_comparison, print_metrics,
# )

# rho_nop = extract_rho(result_spec_nop)
# rho_pi  = extract_rho(result_spec_pi)
#
# # ── Option A: Individual 2D heatmaps ──
# plot_density_matrix_2d(rho_nop, title="Spec = ground, RIP = 800 ns")
# plot_density_matrix_2d(rho_pi,  title="Spec = excited, RIP = 800 ns")
# plt.show()
#
# # ── Option B: Individual 3D bar plots ──
plot_density_matrix_3d(rho_nop, title="Spec = ground, RIP = 800 ns")
# plot_density_matrix_3d(rho_pi,  title="Spec = excited, RIP = 800 ns")
plt.show()
#
# # ── Option C: Side-by-side comparison + difference ──
# figs = plot_comparison(rho_nop, rho_pi, rip_length=800e-9, plot_type="2d")
# plt.show()
#
# # ── Metrics ──
# print_metrics(result_spec_nop, label="Spec = ground")
# print_metrics(result_spec_pi,  label="Spec = excited")

## 결과에서 density matrix 추출 + 저장

반환 구조에서 `analysis_result`를 우선 찾고, wrapper(`.output`)를 반복 언랩해서 안전하게 읽습니다.

최종 추출 항목:
- `rho_hat_real` (4x4 실수부)
- `rho_hat_imag` (4x4 허수부)
- `metrics` (fidelity/purity/min eigenvalue 등)


In [ ]:
# def unwrap_output(obj):
#     cur = obj
#     for _ in range(24):
#         if hasattr(cur, "output"):
#             cur = cur.output
#             continue
#         return cur
#     return cur


# def extract_analysis_output(workflow_result):
#     out = unwrap_output(getattr(workflow_result, "output", workflow_result))

#     if isinstance(out, dict):
#         nested = out.get("analysis_result")
#         if isinstance(nested, dict):
#             return nested
#         if isinstance(out, dict) and "rho_hat_real" in out and "rho_hat_imag" in out:
#             return out

#     # Fallback: task tree search
#     stack = [workflow_result]
#     seen = set()
#     while stack:
#         node = stack.pop()
#         if node is None:
#             continue
#         nid = id(node)
#         if nid in seen:
#             continue
#         seen.add(nid)

#         tasks = getattr(node, "tasks", None)
#         if tasks is not None:
#             try:
#                 task_list = list(tasks)
#             except Exception:
#                 task_list = []
#             names = {getattr(t, "name", "") for t in task_list}
#             if (
#                 "maximum_likelihood_reconstruct" in names
#                 and "extract_assignment_matrix" in names
#             ):
#                 out2 = unwrap_output(getattr(node, "output", None))
#                 if isinstance(out2, dict):
#                     return out2

#             for t in task_list:
#                 stack.append(t)
#                 stack.append(getattr(t, "output", None))

#         stack.append(getattr(node, "output", None))

#     raise RuntimeError("Could not materialize analysis output.")


# analysis = extract_analysis_output(twoq_qst_result)
# rho_real = np.asarray(analysis["rho_hat_real"], dtype=float)
# rho_imag = np.asarray(analysis["rho_hat_imag"], dtype=float)
# rho_hat = rho_real + 1j * rho_imag

# # numerical cleanup
# rho_hat = (rho_hat + rho_hat.conj().T) / 2.0
# rho_hat /= np.trace(rho_hat)

# print("rho_hat shape:", rho_hat.shape)
# print("trace(rho_hat):", np.trace(rho_hat))
# print("metrics:")
# pprint(analysis.get("metrics", {}))

# # Save density matrix
# save_dir = project_root / "examples" / "selectiveRIP" / "output"
# save_dir.mkdir(parents=True, exist_ok=True)
# out_npz = save_dir / "twoq_qst_rho_hat.npz"
# out_json = save_dir / "twoq_qst_metrics.json"

# np.savez(out_npz, rho_hat_real=rho_real, rho_hat_imag=rho_imag, rho_hat=rho_hat)
# out_json.write_text(__import__("json").dumps(analysis.get("metrics", {}), indent=2, ensure_ascii=False))

# print("saved:", out_npz)
# print("saved:", out_json)


# 3Q Quantum State Tomography (RIP + Readout Mitigation)

In [ ]:
from qubit_experiment.experiments import three_qubit_readout_calibration, threeq_qst


## 3Q Readout Calibration

Run once and reuse for tomography sweeps.

## 3Q Tomography Run

## GHZ state

## Result Helpers (3Q)

These helpers make workflow outputs robust in notebook execution order.

## Quick Metrics + Density Matrix Preview (3Q)

# SAVE QPU

In [ ]:
# from laboneq.serializers import save, load, from_dict, from_json, to_dict, to_json
# import time

# t = time.localtime()
# timestamp = time.strftime('%Y%m%d-%H%M', t)

# filename = "RIP"
# save(qpu, filename=f"./qpu_parameters/{timestamp}_{filename}")
